# Moduł 10: Authorization - Role-Based Access Control

**Ćwiczenie:** #11 - Authorization (Permissions)

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Role-Based Access Control](#2-role-based-access-control)
3. [Role Dependencies](#3-role-dependencies)
4. [Resource Ownership](#4-resource-ownership)
5. [Admin-Only Endpoints](#5-admin-only-endpoints)
6. [Permission Systems](#6-permission-systems)
7. [Best Practices](#7-best-practices)
8. [Demo Live Coding](#8-demo-live-coding)
9. [Przygotowanie do ćwiczenia](#9-przygotowanie-do-ćwiczenia)
10. [Podsumowanie](#10-podsumowanie)

---

## 1. Wprowadzenie

### Authentication vs Authorization

**Authentication (Moduł 09) - "Kim jesteś?"**
- Sprawdzasz tożsamość użytkownika
- Login + password → JWT token
- Endpoint: `/auth/login`

**Authorization (Ten moduł) - "Co możesz robić?"**
- Sprawdzasz uprawnienia użytkownika
- Czy user może wykonać akcję?
- Endpoint: `DELETE /posts/5` (tylko autor lub admin)

| Aspekt | Authentication | Authorization |
|--------|----------------|---------------|
| **Pytanie** | Kim jesteś? | Co możesz zrobić? |
| **Odpowiedź** | Użytkownik "john" | Możesz edytować tylko swoje posty |
| **Mechanizm** | Login + JWT | Role + Permissions |
| **Rezultat** | 401 Unauthorized | 403 Forbidden |

**Przykład:**
```
DELETE /posts/5

1. Authentication: Czy user jest zalogowany? (401 jeśli nie)
2. Authorization: Czy user może usunąć post #5? (403 jeśli nie)
```

### Poziomy kontroli dostępu

**Poziom 1: Zalogowany vs niezalogowany**
```python
# Każdy zalogowany może wywołać
@app.get("/posts")
def get_posts(current_user: User = Depends(get_current_user)):
    pass
```

**Poziom 2: Właściciel zasobu**
```python
# Tylko właściciel może edytować
@app.put("/posts/{id}")
def update_post(post_id: int, current_user: User = Depends(get_current_user)):
    post = get_post(post_id)
    if post.user_id != current_user.id:
        raise HTTPException(403, "Not authorized")
```

**Poziom 3: Role (admin vs user)**
```python
# Tylko admin może wywołać
@app.delete("/users/{id}")
def delete_user(current_user: User = Depends(require_admin)):
    pass
```

---

## 2. Role-Based Access Control

### Model User z rolą

**Dodaj pole `role` do modelu User:**

In [ ]:
from sqlalchemy import String, Enum as SQLEnum
from sqlalchemy.orm import Mapped, mapped_column
from enum import Enum
from database import Base

class UserRole(str, Enum):
    """Role użytkownika"""
    USER = "user"
    ADMIN = "admin"

class User(Base):
    __tablename__ = "users"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str] = mapped_column(String(50), unique=True)
    email: Mapped[str] = mapped_column(String(100))
    password_hash: Mapped[str] = mapped_column(String(255))
    role: Mapped[str] = mapped_column(
        SQLEnum(UserRole, native_enum=False),
        default=UserRole.USER  # Domyślnie zwykły user
    )
    
    def is_admin(self) -> bool:
        """Sprawdź czy user jest adminem"""
        return self.role == UserRole.ADMIN

**SQL:**
```sql
ALTER TABLE users ADD COLUMN role VARCHAR(10) DEFAULT 'user';
```

### JWT payload z rolą

**Dodaj rolę do tokenu:**

In [ ]:
# auth.py
def create_access_token(data: dict) -> str:
    """
    Stwórz JWT token
    
    Przykład data:
    {
        "sub": "john",      # Username
        "role": "admin"     # Role (opcjonalne)
    }
    """
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(minutes=30)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

# W endpoincie login:
@app.post("/auth/login")
def login(form_data: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    user = authenticate_user(db, form_data.username, form_data.password)
    if not user:
        raise HTTPException(401, "Incorrect username or password")
    
    # Token z rolą
    access_token = create_access_token(
        data={"sub": user.username, "role": user.role}  # ← Dodaj rolę!
    )
    
    return {"access_token": access_token, "token_type": "bearer"}

**Payload tokenu:**
```json
{
  "sub": "john",
  "role": "admin",
  "exp": 1699999999
}
```

**Dlaczego role w tokenie?**
- ✅ Szybsze - nie trzeba query do bazy przy każdym request
- ✅ Stateless - token zawiera wszystko
- ⚠️ Zmiana roli wymaga nowego tokenu (stary nadal ważny do expiration!)

---

## 3. Role Dependencies

### require_role() dependency

In [ ]:
from fastapi import Depends, HTTPException, status

def require_role(required_role: UserRole):
    """
    Factory function - tworzy dependency sprawdzającą rolę
    
    Args:
        required_role: Wymagana rola (UserRole.USER lub UserRole.ADMIN)
    
    Returns:
        Dependency function
    """
    def role_checker(current_user: User = Depends(get_current_user)) -> User:
        """
        Sprawdź czy user ma wymaganą rolę
        
        Raises:
            403 Forbidden jeśli brak uprawnień
        """
        if current_user.role != required_role:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Access denied. Required role: {required_role}"
            )
        return current_user
    
    return role_checker

### Użycie require_role()

In [ ]:
@app.delete("/users/{user_id}")
def delete_user(
    user_id: int,
    current_user: User = Depends(require_role(UserRole.ADMIN)),  # ← Tylko admin!
    db: Session = Depends(get_db)
):
    """
    Usuń użytkownika (tylko admin)
    
    Returns:
        204 No Content
    
    Raises:
        403 jeśli nie admin
        404 jeśli user nie istnieje
    """
    user = db.get(User, user_id)
    if not user:
        raise HTTPException(404, "User not found")
    
    db.delete(user)
    db.commit()
    return

**Test:**
```bash
# User (nie admin) próbuje usunąć
curl -X DELETE http://localhost:8000/users/1 \
  -H "Authorization: Bearer <user_token>"
# → 403 Forbidden: "Access denied. Required role: admin"

# Admin może usunąć
curl -X DELETE http://localhost:8000/users/1 \
  -H "Authorization: Bearer <admin_token>"
# → 204 No Content
```

### require_admin() - skrót

In [ ]:
def require_admin(current_user: User = Depends(get_current_user)) -> User:
    """
    Dependency: Wymaga roli admin
    
    Skrót dla require_role(UserRole.ADMIN)
    """
    if not current_user.is_admin():
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Admin access required"
        )
    return current_user

# Użycie
@app.get("/admin/stats")
def get_admin_stats(current_user: User = Depends(require_admin)):
    """Panel admina - tylko dla adminów"""
    return {"message": f"Hello admin {current_user.username}"}

---

## 4. Resource Ownership

### Problem: Kto może edytować zasób?

**Zasady:**
- **Właściciel** może edytować swoje zasoby
- **Admin** może edytować wszystkie zasoby
- **Inny user** nie może edytować cudzych zasobów

### Dependency: require_owner_or_admin()

In [ ]:
def require_owner_or_admin(resource_user_id: int):
    """
    Factory: Sprawdź czy user jest właścicielem zasobu lub adminem
    
    Args:
        resource_user_id: ID właściciela zasobu (np. post.user_id)
    """
    def checker(current_user: User = Depends(get_current_user)) -> User:
        # Admin może wszystko
        if current_user.is_admin():
            return current_user
        
        # Właściciel może edytować
        if current_user.id == resource_user_id:
            return current_user
        
        # Inny user - brak dostępu
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="You can only modify your own resources"
        )
    
    return checker

### Użycie w endpoincie

**Problem:** Dependency potrzebuje `resource_user_id`, ale to musimy najpierw pobrać z bazy!

In [ ]:
# ❌ Nie zadziała - post_id jeszcze nie pobrane!
@app.delete("/posts/{post_id}")
def delete_post(
    post_id: int,
    current_user: User = Depends(require_owner_or_admin(???)),  # Skąd wziąć post.user_id?
    db: Session = Depends(get_db)
):
    pass

**Rozwiązanie: Sprawdź wewnątrz endpointu**

In [ ]:
@app.delete("/posts/{post_id}", status_code=204)
def delete_post(
    post_id: int,
    current_user: User = Depends(get_current_user),
    db: Session = Depends(get_db)
):
    """
    Usuń post (właściciel lub admin)
    
    Returns:
        204 No Content
    
    Raises:
        403 jeśli nie właściciel i nie admin
        404 jeśli post nie istnieje
    """
    # 1. Pobierz post
    post = db.get(Post, post_id)
    if not post:
        raise HTTPException(404, "Post not found")
    
    # 2. Sprawdź uprawnienia
    if not current_user.is_admin() and post.user_id != current_user.id:
        raise HTTPException(
            status_code=403,
            detail="You can only delete your own posts"
        )
    
    # 3. Usuń
    db.delete(post)
    db.commit()
    return

**Test:**
```bash
# User "john" (id=1) tworzy post
POST /posts {"title": "My post"}
# → {"id": 5, "user_id": 1, ...}

# John może usunąć swój post
DELETE /posts/5  (Authorization: john's token)
# → 204 No Content ✅

# Alice (id=2) NIE MOŻE usunąć posta Johna
DELETE /posts/5  (Authorization: alice's token)
# → 403 Forbidden ❌

# Admin MOŻE usunąć post Johna
DELETE /posts/5  (Authorization: admin's token)
# → 204 No Content ✅
```

### Reusable helper function

In [ ]:
def check_ownership_or_admin(current_user: User, resource_user_id: int) -> None:
    """
    Sprawdź czy user może modyfikować zasób
    
    Args:
        current_user: Zalogowany user
        resource_user_id: ID właściciela zasobu
    
    Raises:
        403 jeśli nie właściciel i nie admin
    """
    if current_user.is_admin():
        return  # Admin może wszystko
    
    if current_user.id == resource_user_id:
        return  # Właściciel może
    
    raise HTTPException(
        status_code=403,
        detail="You can only modify your own resources"
    )

# Użycie
@app.delete("/posts/{post_id}")
def delete_post(post_id: int, current_user: User = Depends(get_current_user), db: Session = Depends(get_db)):
    post = db.get(Post, post_id)
    if not post:
        raise HTTPException(404, "Post not found")
    
    check_ownership_or_admin(current_user, post.user_id)  # ← Helper!
    
    db.delete(post)
    db.commit()
    return

---

## 5. Admin-Only Endpoints

### Pattern: /admin/* routes

In [ ]:
from fastapi import APIRouter

# Osobny router dla admin endpoints
admin_router = APIRouter(
    prefix="/admin",
    tags=["admin"],
    dependencies=[Depends(require_admin)]  # ← Wszystkie endpointy wymagają admin!
)

@admin_router.get("/users")
def get_all_users(db: Session = Depends(get_db)):
    """Lista wszystkich userów (tylko admin)"""
    users = db.query(User).all()
    return {"users": users}

@admin_router.delete("/users/{user_id}")
def delete_user_admin(user_id: int, db: Session = Depends(get_db)):
    """Usuń usera (tylko admin)"""
    user = db.get(User, user_id)
    if not user:
        raise HTTPException(404, "User not found")
    db.delete(user)
    db.commit()
    return {"message": f"User {user_id} deleted"}

@admin_router.get("/stats")
def get_stats(db: Session = Depends(get_db)):
    """Statystyki systemu (tylko admin)"""
    total_users = db.query(func.count(User.id)).scalar()
    total_posts = db.query(func.count(Post.id)).scalar()
    return {
        "total_users": total_users,
        "total_posts": total_posts
    }

# Include router w main app
app.include_router(admin_router)

**Korzyści:**
- ✅ Wszystkie `/admin/*` endpointy automatycznie wymagają admin
- ✅ Łatwo znaleźć admin endpoints (tag="admin")
- ✅ DRY - dependency na poziomie routera

**Endpointy:**
```
GET    /admin/users       # Lista userów
DELETE /admin/users/{id}  # Usuń usera
GET    /admin/stats       # Statystyki
```

---

## 6. Permission Systems

### Bardziej zaawansowane: Permissions

**Zamiast 2 ról (user/admin), możemy mieć granularne uprawnienia:**

In [ ]:
from enum import Enum

class Permission(str, Enum):
    READ_POSTS = "read:posts"
    WRITE_POSTS = "write:posts"
    DELETE_POSTS = "delete:posts"
    MANAGE_USERS = "manage:users"

# User z listą permissions
class User(Base):
    # ...
    permissions: Mapped[str] = mapped_column(String(255))  # JSON array jako string
    
    def has_permission(self, permission: Permission) -> bool:
        """Sprawdź czy user ma uprawnienie"""
        import json
        perms = json.loads(self.permissions) if self.permissions else []
        return permission in perms

# Dependency
def require_permission(permission: Permission):
    def permission_checker(current_user: User = Depends(get_current_user)) -> User:
        if not current_user.has_permission(permission):
            raise HTTPException(
                status_code=403,
                detail=f"Permission '{permission}' required"
            )
        return current_user
    return permission_checker

# Użycie
@app.delete("/posts/{id}")
def delete_post(
    post_id: int,
    current_user: User = Depends(require_permission(Permission.DELETE_POSTS)),
    db: Session = Depends(get_db)
):
    pass

**Permissions w JWT payload:**
```json
{
  "sub": "john",
  "permissions": ["read:posts", "write:posts", "delete:posts"]
}
```

**Uwaga:** Dla prostoty, w ćwiczeniu #11 używamy tylko ról (user/admin), nie permissions.

---

## 7. Best Practices

### ✅ Rób tak:

**1. Rozróżniaj 401 vs 403:**
```python
# 401 Unauthorized - brak tokenu lub nieprawidłowy token
if not token:
    raise HTTPException(401, "Authentication required")

# 403 Forbidden - user zalogowany, ale brak uprawnień
if not current_user.is_admin():
    raise HTTPException(403, "Admin access required")
```

**2. Domyślnie USER, admin ręcznie:**
```python
# ✅ Dobre - domyślnie user
role: Mapped[str] = mapped_column(default=UserRole.USER)

# Admin ustawiany ręcznie (np. przez seed script)
admin_user = User(username="admin", role=UserRole.ADMIN)
```

**3. Osobny router dla admin:**
```python
# ✅ Dobre
admin_router = APIRouter(
    prefix="/admin",
    dependencies=[Depends(require_admin)]
)

# ❌ Unikaj - duplikacja
@app.get("/endpoint1")
def e1(u = Depends(require_admin)): pass

@app.get("/endpoint2")
def e2(u = Depends(require_admin)): pass
```

**4. Sprawdzaj ownership PRZED akcją:**
```python
# ✅ Dobre
@app.delete("/posts/{id}")
def delete_post(...):
    post = db.get(Post, post_id)
    if not post:
        raise HTTPException(404)  # Najpierw 404
    
    check_ownership_or_admin(current_user, post.user_id)  # Potem 403
    
    db.delete(post)
```

**5. Jasne error messages:**
```python
# ✅ Dobre - jasny komunikat
raise HTTPException(403, "You can only delete your own posts")

# ❌ Złe - za ogólne
raise HTTPException(403, "Forbidden")
```

**6. Admin może wszystko:**
```python
# ✅ Dobre - admin bypasses ownership check
if current_user.is_admin():
    # Admin może wszystko, skip checks
    pass
elif current_user.id != resource.user_id:
    raise HTTPException(403)
```

### ❌ Nie rób tak:

**1. Nie hardcoduj ról w endpointach:**
```python
# ❌ Złe - hardcoded role check
@app.delete("/posts/{id}")
def delete_post(current_user: User = Depends(get_current_user)):
    if current_user.role != "admin":  # ❌ Hardcoded string!
        raise HTTPException(403)

# ✅ Dobre - use dependency
@app.delete("/posts/{id}")
def delete_post(current_user: User = Depends(require_admin)):
    pass
```

**2. Nie sprawdzaj roli w JWT bez weryfikacji:**
```python
# ❌ BARDZO ŹLE - user może zmienić role w tokenie!
payload = jwt.decode(token, verify=False)  # ❌ NO VERIFICATION!
if payload["role"] == "admin":
    # User może podrobić token!
    pass

# ✅ Dobre - zawsze weryfikuj signature
payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
```

**3. Nie ujawniaj czy zasób istnieje:**
```python
# ❌ Złe - zdradza czy post istnieje
@app.get("/posts/{id}")
def get_post(post_id: int, current_user: User = Depends(get_current_user)):
    post = db.get(Post, post_id)
    if not current_user.is_admin() and post.user_id != current_user.id:
        raise HTTPException(403, "Not your post")  # ❌ Zdradza że post istnieje!

# ✅ Dobre - zawsze 404
@app.get("/posts/{id}")
def get_post(post_id: int, current_user: User = Depends(get_current_user)):
    post = db.get(Post, post_id)
    if not post:
        raise HTTPException(404)  # Nie ma - 404
    if not current_user.is_admin() and post.user_id != current_user.id:
        raise HTTPException(404)  # Nie twój - też 404 (nie 403!)
```

In [ ]:
from sqlalchemy import String, ForeignKey, Enum as SQLEnum
from sqlalchemy.orm import Mapped, mapped_column, relationship
from enum import Enum
from database import Base

class UserRole(str, Enum):
    USER = "user"
    ADMIN = "admin"

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str] = mapped_column(String(50), unique=True)
    password_hash: Mapped[str] = mapped_column(String(255))
    role: Mapped[str] = mapped_column(SQLEnum(UserRole), default=UserRole.USER)
    
    posts: Mapped[list["Post"]] = relationship(back_populates="author")
    
    def is_admin(self) -> bool:
        return self.role == UserRole.ADMIN

class Post(Base):
    __tablename__ = "posts"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    content: Mapped[str]
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    
    author: Mapped["User"] = relationship(back_populates="posts")

**dependencies.py:**

In [ ]:
from fastapi import Depends, HTTPException, status
from models import User, UserRole

def require_admin(current_user: User = Depends(get_current_user)) -> User:
    if not current_user.is_admin():
        raise HTTPException(status.HTTP_403_FORBIDDEN, "Admin access required")
    return current_user

def check_ownership_or_admin(current_user: User, resource_user_id: int) -> None:
    if current_user.is_admin():
        return
    if current_user.id == resource_user_id:
        return
    raise HTTPException(
        status.HTTP_403_FORBIDDEN,
        "You can only modify your own resources"
    )

**main.py:**

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, APIRouter
from sqlalchemy.orm import Session
from database import get_db
from models import User, Post
from dependencies import get_current_user, require_admin, check_ownership_or_admin

app = FastAPI(title="Blog API with RBAC")

# --- Public endpoints ---
@app.get("/posts")
def get_posts(db: Session = Depends(get_db)):
    """Lista postów (publiczne)"""
    posts = db.query(Post).all()
    return {"posts": posts}

# --- Authenticated endpoints ---
@app.post("/posts", status_code=201)
def create_post(
    title: str,
    content: str,
    current_user: User = Depends(get_current_user),
    db: Session = Depends(get_db)
):
    """Stwórz post (każdy zalogowany)"""
    post = Post(title=title, content=content, user_id=current_user.id)
    db.add(post)
    db.commit()
    db.refresh(post)
    return post

@app.put("/posts/{post_id}")
def update_post(
    post_id: int,
    title: str,
    content: str,
    current_user: User = Depends(get_current_user),
    db: Session = Depends(get_db)
):
    """Zaktualizuj post (właściciel lub admin)"""
    post = db.get(Post, post_id)
    if not post:
        raise HTTPException(404, "Post not found")
    
    check_ownership_or_admin(current_user, post.user_id)
    
    post.title = title
    post.content = content
    db.commit()
    db.refresh(post)
    return post

@app.delete("/posts/{post_id}", status_code=204)
def delete_post(
    post_id: int,
    current_user: User = Depends(get_current_user),
    db: Session = Depends(get_db)
):
    """Usuń post (właściciel lub admin)"""
    post = db.get(Post, post_id)
    if not post:
        raise HTTPException(404, "Post not found")
    
    check_ownership_or_admin(current_user, post.user_id)
    
    db.delete(post)
    db.commit()
    return

# --- Admin endpoints ---
admin_router = APIRouter(
    prefix="/admin",
    tags=["admin"],
    dependencies=[Depends(require_admin)]
)

@admin_router.get("/users")
def get_all_users(db: Session = Depends(get_db)):
    """Lista wszystkich userów (tylko admin)"""
    users = db.query(User).all()
    return {"users": [{"id": u.id, "username": u.username, "role": u.role} for u in users]}

@admin_router.delete("/users/{user_id}")
def delete_user(user_id: int, db: Session = Depends(get_db)):
    """Usuń usera (tylko admin)"""
    user = db.get(User, user_id)
    if not user:
        raise HTTPException(404, "User not found")
    db.delete(user)
    db.commit()
    return {"message": f"User {user_id} deleted"}

app.include_router(admin_router)

**Test scenarios:**
```bash
# Scenario 1: User tworzy i edytuje swój post
# Login as user "john"
TOKEN_JOHN="..."

# Stwórz post
curl -X POST http://localhost:8000/posts \
  -H "Authorization: Bearer $TOKEN_JOHN" \
  -d "title=My Post&content=Hello"
# → 201 Created {"id": 1, "user_id": 1, ...}

# Edytuj swój post
curl -X PUT http://localhost:8000/posts/1 \
  -H "Authorization: Bearer $TOKEN_JOHN" \
  -d "title=Updated&content=New content"
# → 200 OK ✅

# Scenario 2: User "alice" próbuje edytować post Johna
TOKEN_ALICE="..."

curl -X PUT http://localhost:8000/posts/1 \
  -H "Authorization: Bearer $TOKEN_ALICE" \
  -d "title=Hacked&content=Haha"
# → 403 Forbidden ❌ "You can only modify your own resources"

# Scenario 3: Admin może edytować post Johna
TOKEN_ADMIN="..."

curl -X PUT http://localhost:8000/posts/1 \
  -H "Authorization: Bearer $TOKEN_ADMIN" \
  -d "title=Admin edit&content=Fixed"
# → 200 OK ✅ (admin może wszystko)

# Scenario 4: Only admin can access /admin/users
curl http://localhost:8000/admin/users \
  -H "Authorization: Bearer $TOKEN_JOHN"
# → 403 Forbidden ❌

curl http://localhost:8000/admin/users \
  -H "Authorization: Bearer $TOKEN_ADMIN"
# → 200 OK ✅ {"users": [...]}
```

---

## 10. Podsumowanie

### Kluczowe zagadnienia:

1. **Authentication vs Authorization** - "Kim jesteś?" vs "Co możesz?"
2. **RBAC (Role-Based Access Control)** - user vs admin
3. **Role Dependencies** - require_admin(), require_role()
4. **Resource Ownership** - user może edytować tylko swoje
5. **Admin Override** - admin może wszystko
6. **Admin Router** - osobny prefix dla admin endpoints
7. **401 vs 403** - Unauthorized vs Forbidden

### Poziomy kontroli dostępu:

```python
# 1. Public (niezalogowani)
@app.get("/posts")
def get_posts(): pass

# 2. Authenticated (każdy zalogowany)
@app.post("/posts")
def create_post(current_user: User = Depends(get_current_user)): pass

# 3. Owner or Admin (właściciel lub admin)
@app.delete("/posts/{id}")
def delete_post(...):
    check_ownership_or_admin(current_user, post.user_id)

# 4. Admin only (tylko admin)
@app.delete("/users/{id}")
def delete_user(current_user: User = Depends(require_admin)): pass
```

### Następny krok:
Przejdź do **Moduł 11: Testing** - testowanie API z pytest
